# VisID on Colab (A100)

Runtime → Change runtime type → **A100 GPU**. Then run the cells in order.

Everything that takes time (raw data, images, embeddings, IDs, results) lives on **Google Drive**,
and `run_resumable.sh` skips any stage whose output already exists. If the session disconnects,
reconnect and run all cells again: finished runs are not repeated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/visid'   # persistent home for data + results
!mkdir -p $DRIVE/data/raw $DRIVE/results

Upload the repo to `MyDrive/visid/`, either as `visid.zip` or as the unzipped folder. Then:

In [ ]:
import glob, os, shutil, subprocess
os.chdir('/content')
shutil.rmtree('/content/visid', ignore_errors=True)
zips = glob.glob(f'{DRIVE}/visid.zip')
if zips:
    subprocess.run(['unzip', '-q', zips[0], '-d', '/content'], check=True)
else:  # unzipped folder uploaded: find the repo root (the folder that has configs/base.yaml)
    roots = sorted({os.path.dirname(os.path.dirname(p))
                    for p in glob.glob(f'{DRIVE}/**/configs/base.yaml', recursive=True)})
    assert roots, f'no repo found under {DRIVE}: upload visid.zip or the unzipped visid folder there'
    print('repo found at', roots[0])
    root = os.path.normpath(roots[0])
    def skip(d, names):  # skip only the TOP-LEVEL data/ and results/ (they live on Drive), never visid/data/
        # compare path strings: os.path.samefile is unreliable on the Google Drive mount
        top = {'data', 'results'} if os.path.normpath(d) == root else set()
        return [n for n in names if n in top or n == '__pycache__']
    shutil.copytree(root, '/content/visid', ignore=skip)
assert os.path.exists('/content/visid/visid/data/splits.py'), (
    'visid/data/ (code) is missing from the copy; check that MyDrive/visid has visid/data/splits.py')
os.chdir('/content/visid')
# data/ and results/ point at Drive so they survive disconnects
for d in ('data', 'results'):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    os.symlink(f'{DRIVE}/{d}', d)
os.makedirs(f'{DRIVE}/data/raw', exist_ok=True)
!pip install -q faiss-cpu sentence-transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls && ls visid   # second listing must include data/

**Raw data.** `prepare` tries to download the 2014 Beauty files. If that fails, put
`reviews_Beauty_5.json.gz` and `meta_Beauty.json.gz` in `MyDrive/visid/data/raw/` by hand.

In [ ]:
!python -m pytest -q   # 41 spec tests, ~10 s

**Time it first (optional, ~5 min).** One generator epoch on the sandbox data tells you
the real seconds/epoch on this GPU before committing compute units.

In [ ]:
!python -m visid.cli prepare --config configs/beauty_s3rec_attr.yaml
!python -m visid.cli embed --config configs/beauty_s3rec_attr.yaml --variant attr
!python -m visid.cli build-ids --config configs/beauty_s3rec_attr.yaml --variant attr --seed 0 --set ids.epochs=200
!python -m visid.cli train-tiger --config configs/beauty_s3rec_attr.yaml --variant attr --seed 0 \
    --set tiger.epochs=2 tiger.eval_every=2 tiger.n_enc_layers=4 tiger.n_dec_layers=4 tiger.d_ff=1024 "tiger.cold_eps=[0.0]" tiger.measure_unconstrained=false
!grep -h '"sec"' results/beauty_s3rec/tiger_attr/seed0/train_log.jsonl

**The study.** Seed 0 first (answers the question on its own), then seeds 1 and 2 for mean ± std.
Logs go to Drive too, so you can watch progress from another tab with `!tail` .

In [ ]:
!bash scripts/run_resumable.sh configs/beauty2014.yaml 0 2>&1 | tee -a $DRIVE/run_seed0.log | grep -v -i warning

In [ ]:
!bash scripts/run_resumable.sh configs/beauty2014.yaml 0 1 2 2>&1 | tee -a $DRIVE/run_all.log | grep -v -i warning

In [ ]:
!cat results/beauty2014/table.md
from IPython.display import Image
Image('results/beauty2014/bench/seed0/recall_vs_latency.png')